# RAG-Based Banking Chatbot

A Retrieval-Augmented Generation (RAG) chatbot that answers customer questions about **accounts, loans, credit cards, fees, security, and digital banking** — grounded entirely in a bank's own policy documents, using an **open-source LLM** (no paid API required).

**Stack:** Python · LlamaIndex · Hugging Face Transformers (embeddings + LLM) · Local vector store

**Why RAG?** Instead of relying on the LLM's parametric memory (which can hallucinate rates, fees, and policies), this system retrieves the most relevant policy snippets from a knowledge base at query time and forces the model to answer *only* from that retrieved context — a pattern directly applicable to real banking compliance requirements.

---

## Architecture

```
User question
     │
     ▼
[1] Embed question  ──(sentence-transformers/all-MiniLM-L6-v2)
     │
     ▼
[2] Vector similarity search over 26 banking policy documents
     │
     ▼
[3] Top-k relevant chunks retrieved (k=3)
     │
     ▼
[4] Chunks + question inserted into a grounded prompt template
     │
     ▼
[5] Open-source LLM (TinyLlama-1.1B-Chat) generates the answer
     │
     ▼
Answer + cited sources returned to user
```


## 1. Install dependencies

Run this once. (Skip if you already installed `requirements.txt`.)

In [ ]:
%pip install -q llama-index-core llama-index-embeddings-huggingface \
    llama-index-llms-huggingface llama-index-llms-huggingface-api \
    transformers torch sentence-transformers accelerate

## 2. Load the banking knowledge base

The knowledge base (`data/banking_kb.json`) contains 26 short policy documents across 8 categories: Accounts, Loans, Credit Cards, Fees, Security & Compliance, Digital Banking, Interest & Rates, and Customer Service.

In a real deployment, this would be swapped for the bank's actual FAQ pages, product terms & conditions PDFs, and internal policy documents — the pipeline below doesn't change.

In [ ]:
import json
import sys
sys.path.append("src")

with open("data/banking_kb.json") as f:
    records = json.load(f)

print(f"Loaded {len(records)} knowledge base documents")
categories = sorted(set(r["category"] for r in records))
print("Categories:", categories)
records[0]

## 3. Build the document set and chunk it

In [ ]:
from llama_index.core import Document, Settings
from llama_index.core.node_parser import SentenceSplitter

documents = [
    Document(
        text=f"{r['title']}\n\n{r['content']}",
        doc_id=r["id"],
        metadata={"id": r["id"], "category": r["category"], "title": r["title"]},
    )
    for r in records
]

Settings.node_parser = SentenceSplitter(chunk_size=512, chunk_overlap=50)
print(f"Prepared {len(documents)} documents for indexing")

## 4. Load the embedding model and build the vector index

We use `sentence-transformers/all-MiniLM-L6-v2` — a small (~80MB), fast, open-source embedding model that runs comfortably on CPU and produces 384-dimensional embeddings.

In [ ]:
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core import VectorStoreIndex

embed_model = HuggingFaceEmbedding(model_name="sentence-transformers/all-MiniLM-L6-v2")
Settings.embed_model = embed_model

index = VectorStoreIndex.from_documents(documents, show_progress=True)

# Persist so we don't need to re-embed on every run
index.storage_context.persist(persist_dir="storage")
print("Vector index built and saved to ./storage")

## 5. Load the open-source LLM

`TinyLlama-1.1B-Chat` is small enough to run on a laptop CPU (with patience) or a modest GPU. For noticeably better answer quality, swap `LLM_MODEL_NAME` in `src/config.py` for a larger model such as `Qwen/Qwen2.5-7B-Instruct` or `mistralai/Mistral-7B-Instruct-v0.3` if you have a GPU with ≥12GB VRAM.

**No GPU?** Set `USE_HF_INFERENCE_API = True` in `src/config.py` and export a free Hugging Face API token — this calls a hosted model instead of downloading weights locally.

In [ ]:
from llama_index.llms.huggingface import HuggingFaceLLM

llm = HuggingFaceLLM(
    model_name="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    tokenizer_name="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    context_window=2048,
    max_new_tokens=256,
    generate_kwargs={"temperature": 0.1, "do_sample": True},
    device_map="auto",
)
Settings.llm = llm
print("LLM loaded")

## 6. Build the grounded query engine

The prompt template below is the most important piece of the whole system: it instructs the model to answer **only** from retrieved context and to explicitly say when it doesn't know something, rather than inventing a fee or rate — critical for a banking use case.

In [ ]:
from llama_index.core import PromptTemplate

SYSTEM_PROMPT = (
    "You are a helpful, precise banking assistant for a retail bank. "
    "Answer the customer's question using ONLY the information given in the "
    "context below. If the answer is not contained in the context, say "
    "'I don't have that information — please contact customer service.' "
    "Do not make up policies, numbers, or fees. Keep answers concise and "
    "cite specific figures (fees, rates, timeframes) from the context when "
    "relevant."
)

QA_TEMPLATE = PromptTemplate(
    SYSTEM_PROMPT
    + "\n\n---------------------\n"
    + "Context:\n{context_str}\n"
    + "---------------------\n\n"
    + "Question: {query_str}\n"
    + "Answer: "
)

query_engine = index.as_query_engine(similarity_top_k=3, text_qa_template=QA_TEMPLATE)
print("Query engine ready")

## 7. Ask questions 🎉

In [ ]:
def ask(question):
    response = query_engine.query(question)
    print("Q:", question)
    print("\nA:", str(response))
    print("\nSources used:")
    for node in response.source_nodes:
        print(f"  - {node.metadata.get('title')} (score: {round(node.score, 3)})")
    print("\n" + "-"*80)
    return response

ask("What documents do I need to open a savings account?")

In [ ]:
ask("How much is the overdraft fee and how many times can I be charged per day?")

In [ ]:
ask("Can I pay off my personal loan early without a penalty?")

In [ ]:
# A question NOT covered by the knowledge base — the model should say it doesn't know,
# rather than hallucinating an answer. This is the key safety behavior for a banking bot.
ask("What is the CEO's phone number?")

## 8. Interactive chat loop (optional)

Uncomment and run the cell below for a live back-and-forth chat session in the notebook.

In [ ]:
# while True:
#     q = input("You: ")
#     if q.lower() in {"exit", "quit"}:
#         break
#     ask(q)

## 9. Evaluation & next steps

**How I'd evaluate this in production:**
- **Retrieval quality:** hit-rate / MRR on a labeled set of (question → correct source doc_id) pairs
- **Answer faithfulness:** does the generated answer only contain facts present in the retrieved context? (can be scored with an LLM-as-judge or the `ragas` library)
- **Refusal correctness:** does the bot correctly decline out-of-scope questions instead of hallucinating?

**Possible extensions for a stronger portfolio piece:**
- Swap the flat JSON KB for real PDFs (loan agreements, T&Cs) using LlamaIndex's PDF readers
- Add conversation memory (`ChatMemoryBuffer`) for multi-turn context
- Add a re-ranking step (e.g. `bge-reranker`) after initial retrieval for higher precision
- Wrap in a Streamlit/Gradio UI for a live demo link
- Add guardrails (e.g. NeMo Guardrails or simple regex/PII filters) before responses reach the user
- Deploy the vector index in a proper vector DB (Chroma, Qdrant, Pinecone) instead of local storage for scale

---

*This project demonstrates: RAG architecture design, open-source LLM integration, prompt engineering for factual grounding, and awareness of production concerns like hallucination control — all directly relevant to applied AI/ML roles in fintech.*